In [ ]:
import pandas as pd

# Path to data frame with WSIs
df_path = r"D:\DATA\abmil_exp3.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to output dir
output_path = r"D:\NOTEBOOKS\Christine\all_slides\combined_feature_summary.csv"

# Path to zarr
zarr_dir = r"Q:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
class_dict = {
    'Normal Tissue': 0, 
    'Morphology Not Applicable / Insufficient Tissue': 0, 
    'Cellular Changes / Abnormal Tissue Structure': 1,
    'Traumatic Lesions': 1, 
    'Congenital Malformations': 1,
    'Pregnancy-Related Tissues/Changes': 1,
    'Obstruction / Fluid Retention / Cysts': 1, 
    'Mechanical Changes / Architectural Distortion': 1,
    'Inflammation': 2, 
    'Fibrosis': 1, 
    'Degeneration / Necrosis / Atrophy': 1, 
    'Material Deposits': 1, 
    'Resection Margin Free': 0, 
    'Resection Margin Uncertain': 3,
    'Resection Margin Not Free': 4, 
    'Proliferative/Pre-neoplastic Changes': 3, 
    'Benign Neoplasm': 3, 
    'Uncertain / Borderline Neoplasm': 3, 
    'In Situ Neoplasm': 4, 
    'Malignant Neoplasm': 4,
}

df_all["M_idx"] = df_all["M_category"].apply(
    lambda lst: [class_dict[x] for x in lst]
)

# 0: Normal
# 1: Other morphologies
# 2: Inflammation
# 3: Neoplastic Changes, Benign/Uncertain/Borderline
# 4: In Situ, Malignant Neoplasm 

In [ ]:
df_sub = df_all.copy()
df_sub['M_idx'] = df_sub['M_idx'].apply(
    lambda x: max(x) if isinstance(x, list) else x
)

In [ ]:
df_sub['M_idx'].value_counts()

In [ ]:
from abmil import KFoldPipeline

pipeline = KFoldPipeline(
    df=df_sub,
    filename_col='filename',
    label_col='M_idx',
    feature_key='features_h-optimus-0',
    tile_key="tiles_224",
    zarr_dir=zarr_dir,
)
pipeline.validate_slides()

In [ ]:
results = pipeline.kfold_cross_validation(n_splits=5, n_epochs=100, max_tiles=50000, resume_from_checkpoints=True, checkpoint_dir="checkpoints/exp3_h-optimus-0")

In [ ]:
pipeline.print_results()

In [ ]:
from abmil import confusion_matrix_report, auc_score, per_class_auc, plot_roc_curve

confusion_matrix_report(all_labels, all_preds)

auc = auc_score(all_labels, all_probs)
print("Macro AUC:", auc)

per_class_auc(all_labels, all_probs)

plot_roc_curve(all_labels, all_probs)

In [ ]:
import os
import numpy as np
import pandas as pd
from wsidata import open_wsi

feature_keys = {
    "H-optimus-0": "features_h-optimus-0"
}

paths = df_sub["filename"].tolist()
label_map = df_sub.groupby("filename")["M_idx"].max().to_dict()

rows = []

for slide_path in paths:
    print(slide_path)
    zarr_path = os.path.join(zarr_dir, os.path.basename(slide_path).replace(".mrxs", ".zarr"))

    try:
        wsi = open_wsi(slide_path, zarr_path)
    except Exception as e:
        print(e)
        continue

    label = label_map.get(slide_path)

    for model, key in feature_keys.items():
        X = wsi.tables.get(key, {}).X if key in wsi.tables else None
        if X is None or X.size == 0:
            continue

        norms = np.linalg.norm(X, axis=1)
        bag_size = X.shape[0]

        rows.extend([
            {
                "filename": slide_path,
                "model": model,
                "M_idx": label,
                "bag_size": bag_size,
                "feature_norm": float(n),
            }
            for n in norms
        ])

df_plot = pd.DataFrame(rows)

In [ ]:
norm_summary = (
    df_plot
    .groupby("model")["feature_norm"]
    .agg(["count", "mean", "std", "median"])
    .round(4)
)

print("Norm Summary:",
      norm_summary)

bag_df = (df_plot[df_plot["model"] == "H-optimus-0"])

bag_summary = (
    bag_df
    .groupby("M_idx")["bag_size"]
    .agg(["count", "mean", "std", "median"])
    .round(2)
)

print("Bag Summary:",
      bag_summary)

In [ ]:
m_idx_map = {
    0: "0 - Normal (n=121)",
    1: "1 - Other (n=223)",
    2: "2 - Inflammation (n=227)",
    3: "3 - Neoplastic changes (Benign/Uncertain/Borderline) (n=265)",
    4: "4 - Neoplasm (In Situ/Malignant) (n=164)"
}
bag_df["M_label"] = bag_df["M_idx"].map(m_idx_map)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

order = ["0 - Normal (n=)", "1 - Morphology not applicable (n=)", "2 - Other (n=)", "3 - Inflammation (n=)", "4 - Neoplastic changes (Benign/Uncertain/Borderline) (n=)", "5 - Neoplasm (In Situ/Malignant)"]

palette = sns.color_palette(n_colors=len(order))
color_map = dict(zip(order, palette))

sns.set_style("white")

fig, ax = plt.subplots(figsize=(7, 5))

sns.histplot(
    data=bag_df,
    x="bag_size",
    hue="M_label",
    hue_order=order,
    bins=50,
    multiple="stack",
    alpha=0.4,
    palette=color_map,
    ax=ax
)

legend = ax.get_legend()
legend.set_title("M idx")

handles = legend.legend_handles
ymax = ax.get_ylim()[1]

for i, label in enumerate(order):
    values = bag_df.loc[bag_df["M_label"] == label, "bag_size"]

    mean_val = values.mean()
    median_val = values.median()
    color = color_map[label]

    ax.axvline(mean_val, color=color, linestyle="-", linewidth=2)
    ax.axvline(median_val, color=color, linestyle="--", linewidth=2)

    # Mean
    ax.text(
        mean_val,
        ymax * 0.9,
        f"Mean: {mean_val:.1f}",
        color=color,
        ha='left',
        fontsize=8, 
        backgroundcolor='white'
    )
    
    # Median
    ax.text(
        median_val,
        ymax * 0.75,
        f"Median: {median_val:.1f}",
        color=color,
        ha='left',
        fontsize=8,
        fontweight='bold',
        backgroundcolor='white'
    )

ax.set_title("Bag Size Distribution")
ax.set_xlabel("Instances per Slide")
ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()

sns.set_style("white")

ax = sns.histplot(
    data=df_plot,
    x="feature_norm",
    hue="model",
    bins=100,
    alpha=0.4
)

sns.move_legend(
    ax,
    "upper left",
    bbox_to_anchor=(1, 1),
    title="Model"
)

plt.title("Feature Norm Distribution per Model")
plt.xlabel("L2 Norm")

plt.tight_layout()
plt.show()

In [ ]:
df_plot["norm_zscore"] = df_plot.groupby(["filename", "model"])["feature_norm"] \
    .transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))

In [ ]:
plt.figure()
sns.histplot(
    data=df_plot,
    x="norm_zscore",
    hue="model",
    bins=100,
    alpha=0.4
)
plt.title("Normalized Feature Norm Distribution")
plt.xlabel("Z-scored Norm")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- (1) Raw norms ---
sns.histplot(
    data=df_plot,
    x="feature_norm",
    hue="model",
    bins=100,
    alpha=0.4,
    ax=axes[0],
    legend= True
)
legend = ax.get_legend()
legend.set_title("M idx")
axes[0].set_title("Feature Norm Distribution")
axes[0].set_xlabel("L2 Norm")

# --- (2) Normalized norms ---
sns.histplot(
    data=df_plot,
    x="norm_zscore",
    hue="model",
    bins=100,
    alpha=0.4,
    ax=axes[1],
)
axes[1].set_title("Normalized Norms (Per-Slide)")
axes[1].set_xlabel("Z-scored Norm")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Aggregate mean feature norm per slide
df_agg = df_plot[df_plot["model"] == "H-optimus-0"].groupby("filename").agg({
    "feature_norm": "mean",
    "bag_size": "first",
    "M_idx": "first"
}).reset_index()

# Map M_idx to labels
m_idx_map = {
    0: "0 - Normal",
    1: "1 - Other",
    2: "2 - Inflammation",
    3: "3 - Neoplastic (Benign/Uncertain)",
    4: "4 - Neoplasm (In Situ/Malignant)"
}
df_agg["M_label"] = df_agg["M_idx"].map(m_idx_map)

# Create scatter plot
fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(
    data=df_agg,
    x="bag_size",
    y="feature_norm",
    hue="M_label",
    s=100,
    alpha=0.6,
    ax=ax
)

ax.set_xlabel("Bag Size (# Tiles per Slide)")
ax.set_ylabel("Mean Feature Norm (L2)")
ax.set_title("Mean Feature Norm vs. Bag Size by M_idx")
ax.legend(title="Morphology Category", bbox_to_anchor=(1.05, 1), loc='upper left')

sns.despine()
plt.tight_layout()
plt.show()
